# Bringing in Imagery: IF Panels and H&E Slides

Xenium provides transcripts and segmentation, but the same tissue section is often imaged with other modalities: a multiplexed immunofluorescence (IF) protein panel, or a plain histology (H&E) stain. Both are just pixels until tied to existing cell boundaries. Once tied, every cell gets new per-channel numeric features (protein intensity, or stain concentration) that behave like any other `obs`/`X` column: color points by them, correlate them with expression, gate on them.

This notebook covers the two image types astrolabe treats specially:

1. **IF** (`io.read_ome_tiff` + `pp.extract_if_intensities`) — multiplexed protein channels, mean intensity per cell.
2. **H&E** (`io.read_he` + `pp.extract_if_intensities(..., if_key='he')`) — the same machinery, plus a stain-deconvolution step that turns RGB into `[hematoxylin, eosin, residual]` concentrations.

**Sample data provenance — both examples below use real, correctly-registered pairings, not fabricated ones:**

- The **IF panel** comes from a *different* Xenium sample than the rest of this notebook: `ab.datasets.xenium_tiny()` (Human Kidney, Xenium Protein panel, ~20 MB) ships real immunofluorescence channels — including four immune-checkpoint stains (PD-1, VISTA, PD-L1, LAG-3) — captured by the instrument itself alongside its own 358 real segmented cells. It's used here specifically *because* it has a real companion IF image, unlike the main lung cancer dataset.
- The **H&E slide** is the real scan `ab.datasets.xenium()` already attaches to the lung cancer dataset as `mdata['he']`, registered via 10x's own `he_imagealignment.csv`.

An earlier version of this notebook downloaded unrelated standalone sample images (`ab.datasets.ome_tiff()`, `ab.datasets.he()`) and stretched them across the lung tissue's bounding box with a fabricated affine purely to exercise the extraction code. That pairing was visually misleading — overlaying an unrelated picture under real cell points looks like it means something biologically when it doesn't — so this notebook now uses two pairings that are genuinely real instead. `ab.datasets.ome_tiff()`/`he()` remain available in the library for other uses (e.g. testing `io.read_ome_tiff` in isolation), just not demonstrated here.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import astrolabe as ab

Two `MuData` objects are needed here for the IF section specifically: `mdata`, the familiar lung cancer sample (`ab.datasets.xenium()`, used for the H&E and morphology sections later), and `if_mdata`, a *separate* small Xenium run (`ab.datasets.xenium_tiny()`) whose own real IF panel we want.

In [ ]:
mdata = ab.datasets.xenium()

# The IF demo below needs a *real* companion immunofluorescence image, not a
# fabricated pairing -- so it uses a different, much smaller Xenium run
# (Human Kidney, Xenium Protein panel) that ships one, rather than stretching
# an unrelated picture across this lung sample's cell layout.
if_mdata = ab.datasets.xenium_tiny()
print(if_mdata)

## 2. Load the real IF panel

Xenium's multimodal-segmentation output includes `morphology_focus/`: one real, multi-channel
OME-XML document split across several single-channel companion TIFF files (a common OME-TIFF
packaging convention). Reading *any one* of them with `io.read_ome_tiff` assembles the full
panel, because tifffile follows the OME-XML's own references to the sibling files sitting next
to it on disk.

No `registration_affine` is needed here: this panel was captured by the Xenium instrument
itself, in the exact same pixel grid as the cell segmentation — the same situation as the
`'morphology'` modality in section 7 below. A `registration_affine` is only required when the
image comes from a genuinely separate scan with its own independent pixel grid, e.g. a
different microscope or a rescan after the Xenium run — see the H&E section for that real case.

In [ ]:
from pathlib import Path

if_lib = list(if_mdata['xenium'].uns['spatial'])[0]
outs_dir = Path(if_mdata['xenium'].uns['spatial'][if_lib]['source_path'])
panel_path = outs_dir / "morphology_focus" / "ch0000_dapi.ome.tif"

ab.io.read_ome_tiff(str(panel_path), mdata=if_mdata, mdata_key="if")
print("channels:", list(if_mdata['if'].var_names))

## 3. Extract per-cell IF intensities

`pp.extract_if_intensities` walks every cell's boundary polygon in `if_mdata['xenium']`,
converts it from microns to image pixels (using `pixel_size_um` alone here, since there's no
`registration_affine`), rasterizes the polygon, and reduces the pixels under it (mean by
default) per channel. The result replaces `if_mdata['if']` with a new `AnnData` whose `X` is
`(n_cells, n_channels)` and whose `obs_names` match the Xenium modality exactly, so the two are
aligned and can be queried together.

In [ ]:
ab.pp.extract_if_intensities(if_mdata, xenium_key="xenium", if_key="if")

print(if_mdata['if'])
print()
print(if_mdata['if'].X[:5])
print()
print("obs_names aligned with xenium:", (if_mdata['if'].obs_names == if_mdata['xenium'].obs_names).all())

Unlike a fabricated pairing, these are genuine per-cell measurements from a real specimen: 358 segmented kidney cells, each with 12 real channel intensities — the DAPI/membrane/mesenchymal markers Xenium's own multimodal segmentation model uses (`DAPI`, `ATP1A1/CD45/E-Cadherin`, `18S`, `alphaSMA/Vimentin`), four unused imaging-round background channels, and four real immune-checkpoint stains (`PD-1`, `VISTA`, `PD-L1`, `LAG-3`). Every cell has a meaningful (non-zero) value for the channels that are actually expressed in its neighborhood, since the image genuinely corresponds to where these cells were segmented — no undersampling artifact to explain away.

With `if_mdata['if'].X` filled, every channel is another numeric feature: copy it
into `if_mdata['xenium'].obs` (or query `if_mdata['if']` directly) and treat it like any QC
metric, e.g. color points by it or gate on it. The overlay below also crops to the small real
tissue footprint (`xenium_tiny` has ~360 cells clustered in one corner of a much larger imaged
field) so both the image and points render at a scale where real texture is visible.

In [ ]:
# Pull two real channels into obs, colored below: a membrane/immune marker
# and a mesenchymal marker -- genuinely distinct kidney cell populations.
if_mdata['xenium'].obs["if_membrane"] = np.asarray(if_mdata['if'][:, "ATP1A1/CD45/E-Cadherin"].X).ravel()
if_mdata['xenium'].obs["if_stromal"] = np.asarray(if_mdata['if'][:, "alphaSMA/Vimentin"].X).ravel()

# pp.auto_crop finds a tight box around the cells on its own -- this tiny
# sample isn't one compact blob (a couple of cells sit far from the main
# cluster), and a naive min/max-based crop would blow the box out to mostly
# empty space just to include a stray outlier. auto_crop uses percentile-
# trimmed bounds instead, so a handful of outliers on either side don't set
# the frame. inplace=True on a copy keeps the 'if' sibling modality (and its
# spatial_bounds viewport, synced automatically) instead of returning just
# the cropped 'xenium' AnnData on its own.
if_crop = if_mdata.copy()
ab.pp.auto_crop(if_crop, padding=0.15, inplace=True)

(
    (
        ab.pl.image(if_crop, modality="if", image_key="full", channel="ATP1A1/CD45/E-Cadherin")
        + ab.pl.points(if_crop, color="if_membrane")
    )
    | (
        ab.pl.boundaries(if_crop, alpha=0.6)
        + ab.pl.points(if_crop, color="if_stromal")
    )
).show()

Check channel distributions before treating them as features: `pl.intensity_histograms` shows one histogram per channel with an Otsu-threshold overlay. A bimodal distribution (one near-zero peak, one positive) is typical for a real protein signal; a unimodal smear often indicates a dim or failed channel. `pl.scatter_gate` cross-plots two channels with Otsu thresholds on both axes to identify single-positive, double-positive, and double-negative cell populations.

In [ ]:
# One histogram per channel of interest, with an Otsu threshold line.
ab.pl.intensity_histograms(if_mdata['if'][:, ["ATP1A1/CD45/E-Cadherin", "alphaSMA/Vimentin", "PD-1", "PD-L1"]]).show()

In [ ]:
# Cross-plot the membrane/immune marker vs. the mesenchymal marker with Otsu
# thresholds on both axes. Each quadrant is labelled; cells in the
# upper-right are double-positive.
ab.pl.scatter_gate(if_mdata['if'], x="ATP1A1/CD45/E-Cadherin", y="alphaSMA/Vimentin").show()

The scatter gate shows a genuine four-population split: roughly 44% of cells are double-negative (neither marker detected), 31% are membrane/immune-marker-positive only, 11% are mesenchymal-marker-positive only, and 15% are double-positive. This is a real, if coarse, tissue-composition readout for this small kidney field: epithelial and immune cells (ATP1A1/CD45/E-Cadherin+) versus stromal cells (alphaSMA/Vimentin+), with a double-positive population plausibly reflecting boundary/transitional cells or segmentation bleed between adjacent cell types. In general, the four quadrants identify biologically distinct populations: double-negative (background or marker-absent cells), single-positive for each channel (cells expressing one target), and double-positive (co-expressing cells, e.g., CD4+CD8+ double-positive thymocytes in other contexts). The Otsu thresholds adapt to the observed intensity distribution, so the gate is data-driven rather than requiring manual cutoff selection.

## 4. H&E: extract stain concentrations from the real, natively-registered scan

`io.read_he` works the same way as `read_ome_tiff`, but for a plain RGB histology image. The
key difference is `separate_stains` (default `True`): a stain deconvolution matrix is estimated
immediately and stored on the `ImageSpec`, so the three "channels" astrolabe exposes are not
raw R/G/B but `['hematoxylin', 'eosin', 'residual']` optical-density concentrations.

Unlike the IF panel above (a separate small Xenium run brought in just for this demo), the H&E
image needs no extra download here: `ab.datasets.xenium()` already attached a **real** H&E scan
of this exact lung tissue section as `mdata['he']`, registered via 10x's own
`he_imagealignment.csv` — a genuine (and, as it happens, roughly-90°-rotated) affine, not a
fabricated one. This is the case a `registration_affine` actually exists for: an independent
scan of the same physical section, at its own pixel grid and orientation.

In [ ]:
he_library_id = list(mdata['he'].uns['spatial'])[0]
he_spec = mdata['he'].uns['spatial'][he_library_id]['images']['full']

print("H&E full_shape:", he_spec.full_shape, " dimension_order:", he_spec.dimension_order)
print("stain_matrix set:", he_spec.stain_matrix is not None)
print()
print("Real registration_affine (from he_imagealignment.csv, not fabricated):")
print(he_spec.registration_affine)

`pp.extract_if_intensities` is the same function used for IF: pass `if_key='he'` and it
detects the `stain_matrix` on the spec and applies pixel-level optical-density deconvolution
*before* averaging within each cell polygon, rather than just averaging raw RGB.

In [ ]:
ab.pp.extract_if_intensities(mdata, xenium_key="xenium", if_key="he")

print(mdata['he'])
print()
print("stain concentrations (first 5 cells):")
print(mdata['he'].X[:5])

These are real per-cell stain concentrations for the actual lung tissue section imaged alongside this Xenium run — not a different, unrelated slide. The first five cells show a realistic mix: both hematoxylin and eosin present at meaningful, cell-to-cell-varying levels (roughly 0.07–0.58 hematoxylin, 0–0.94 eosin), with residual staying near zero as expected for a well-fit deconvolution. In real H&E data, hematoxylin OD is a proxy for nuclear density (compact, hyperchromatic tumor nuclei score high; loose stromal cells score lower) and eosin OD reflects cytoplasmic and matrix content. A higher hematoxylin-to-eosin ratio is characteristic of lymphocytes and other small, dense-nucleus cells; a lower ratio indicates cytoplasm-rich cells like hepatocytes or glandular epithelium. Because this is real, correctly-registered data, these values are directly usable for downstream analysis (e.g. correlating with `obs['cell_area']` or clustering), unlike the earlier fabricated pairing.

In [ ]:
# Same pattern as the IF channel: pull a stain concentration into xenium.obs and plot it.
mdata['xenium'].obs['he_hematoxylin'] = np.asarray(mdata['he'][:, 'hematoxylin'].X).ravel()

(
    ab.pl.image(mdata, modality="he", image_key="full")
    + ab.pl.points(mdata, color="he_hematoxylin")
).show()

## 5. Re-fitting the stain matrix with `pp.estimate_stain_matrix`

`read_he` already estimated a stain matrix using Macenko SVD (the default `method`). If channel
separation looks poor for a particular slide (too much hematoxylin bleeding into the eosin
channel, for instance), `pp.estimate_stain_matrix` re-fits it in place without re-reading
the file, and can try the alternative method (`'ruifrok'`, the fixed published reference matrix,
vs. `'macenko'`, fit to this slide's actual pixel distribution).

In [ ]:
print("Macenko-fit matrix (from read_he):")
print(mdata['he'].uns['spatial'][he_library_id]['images']['full'].stain_matrix)

ab.pp.estimate_stain_matrix(mdata, modality="he", method="ruifrok")
print()
print("Ruifrok-Johnston reference matrix (after estimate_stain_matrix):")
print(mdata['he'].uns['spatial'][he_library_id]['images']['full'].stain_matrix)

# Switch back to Macenko and re-extract so the rest of the notebook uses the
# slide-adaptive fit.
ab.pp.estimate_stain_matrix(mdata, modality="he", method="macenko")
ab.pp.extract_if_intensities(mdata, xenium_key="xenium", if_key="he")

## 6. Fixing a rendered image with `pp.add_image_filter`

Sometimes the raw pixels aren't the problem: the *rendering* is, whether a noisy IF channel,
flat illumination that washes out contrast, or periodic stitching artifacts from a tiled scanner.
`pp.add_image_filter` appends a filter callable to `ImageSpec.filter_chain`; filters run
per-channel (2-D, `float32` in/out) and are applied **both** at render time (`pl.image`) and at
analysis time (`pp.extract_if_intensities`), so a filter added to clean up a plot also cleans
up any per-cell intensities extracted afterward. Multiple calls stack filters in the order
added.

Three filters live in `pl._filters` (re-exported from `pl`):

- **`GaussianFilter(sigma=...)`** — spatial blur, smooths sensor noise.
- **`CLAHEFilter(clip_limit=..., kernel_size=...)`** — local contrast enhancement; pulls
  faint structure out of a low-contrast channel without blowing out bright regions.
- **`FFTFilter(cutoff=..., fov_size_px=...)`** — frequency-domain high-pass for flat-field
  correction, plus an optional notch filter that removes periodic tile-stitching artifacts at
  harmonics of `1 / fov_size_px`.

In [ ]:
# Baseline render of the real membrane/immune marker channel before any filtering.
ab.pl.image(if_crop, modality="if", image_key="full", channel="ATP1A1/CD45/E-Cadherin", title="IF — raw").show()

In [ ]:
# Gaussian blur to knock down pixel-level noise.
ab.pp.add_image_filter(if_crop, ab.pl.GaussianFilter(sigma=1.5), modality="if")

ab.pl.image(if_crop, modality="if", image_key="full", channel="ATP1A1/CD45/E-Cadherin", title="IF — Gaussian-smoothed").show()

In [ ]:
# CLAHE on top of the Gaussian blur — filters stack in the order added, so this
# operates on the already-smoothed channel. Boosts local contrast for faint structure.
ab.pp.add_image_filter(if_crop, ab.pl.CLAHEFilter(clip_limit=0.02), modality="if")

ab.pl.image(if_crop, modality="if", image_key="full", channel="ATP1A1/CD45/E-Cadherin", title="IF — Gaussian + CLAHE").show()

`FFTFilter` is the one to reach for when flat, uneven illumination washes out real contrast
(its `cutoff` parameter), or — for a large tiled/stitched scan, not demonstrated here — when a
tiled scanner leaves periodic artifacts at FOV boundaries (`fov_size_px`). It is demonstrated
below on a fresh, unfiltered IF read so its effect isn't confused with the Gaussian/CLAHE chain
above.

In [ ]:
# Demonstrate FFTFilter independently on a separate, unfiltered read of the same panel,
# cropped to the same viewport as if_crop for a comparable render.
fft_demo_adata = ab.io.read_ome_tiff(str(panel_path))

import mudata as md
fft_mdata = md.MuData({"xenium": if_crop['xenium'].copy(), "if": fft_demo_adata})
fft_mdata.uns['default_modality'] = 'if'
fft_mdata['if'].uns['spatial_bounds'] = dict(if_crop['if'].uns['spatial_bounds'])

ab.pl.image(fft_mdata, modality="if", image_key="full", channel="ATP1A1/CD45/E-Cadherin", title="IF — before FFTFilter").show()

# cutoff=0.05 applies a flat-field high-pass, revealing the membrane texture
# underneath a slowly-varying illumination gradient. fov_size_px (the notch
# component that removes periodic tile-stitching artifacts) is deliberately
# left unset here: this crop is a single small field from one real scan, not
# a large stitched mosaic, so it has no such periodic artifact to remove --
# notching out harmonics of an invented FOV size would instead zero out real
# spatial frequencies at that scale, producing meaningless banding rather
# than a cleaner image. See the docstring for a stitched-scan example.
ab.pp.add_image_filter(fft_mdata, ab.pl.FFTFilter(cutoff=0.05), modality="if")

ab.pl.image(fft_mdata, modality="if", image_key="full", channel="ATP1A1/CD45/E-Cadherin", title="IF — after FFTFilter").show()

Filters extracted with `pp.extract_if_intensities` see the same processed pixels as the
render above: re-extracting after adding a filter changes the per-cell numbers, not just the
picture.

In [ ]:
before = if_crop['if'].X.copy()
ab.pp.extract_if_intensities(if_crop, xenium_key="xenium", if_key="if")
after = if_crop['if'].X

print("Mean channel intensity, unfiltered extraction vs. with Gaussian+CLAHE filter_chain:")
print("  before:", before.mean(axis=0))
print("  after: ", after.mean(axis=0))

The rendered channel (`ATP1A1/CD45/E-Cadherin`) shows the effect clearly: mean intensity nearly quadruples (~31.8 → ~119.9, +277%). Gaussian blur alone would only redistribute local values toward neighborhood averages; the large jump comes from CLAHE's local contrast stretch pulling up the real, spatially-structured membrane texture visible in the image above — genuine tissue detail that a flat percentile stretch alone would leave compressed near zero. Other channels shift by different amounts (some barely move, one — `PD-1` — actually drops slightly), because CLAHE's effect depends on each channel's own local intensity structure, not a single global scale factor. The key point is that filters affect both visualization *and* downstream extraction: check `if_crop['if'].X` after adding each filter to confirm the quantitative effect is as intended before continuing to downstream analysis.

## Summary

- `io.read_ome_tiff` + `pp.extract_if_intensities` turn a multiplexed IF panel into per-cell
  protein channels in `mdata['if'].X`, aligned to the Xenium modality's `obs_names`.
- `io.read_he` + `pp.extract_if_intensities(..., if_key='he')` do the same for H&E, with an
  extra stain-deconvolution step (`pp.estimate_stain_matrix` to re-fit it) that turns RGB into
  `[hematoxylin, eosin, residual]` concentrations.
- A `registration_affine` maps image pixels into the same micron frame as the segmentation
  boundaries — but only when the image comes from a genuinely separate scan with its own pixel
  grid (like the H&E slide's real `he_imagealignment.csv` affine here). An image captured by the
  Xenium instrument itself in the same run (like the IF panel here, or the `'morphology'`
  modality in section 7) needs no affine at all: `pixel_size_um` alone is enough.
- `pp.add_image_filter` with `GaussianFilter`, `CLAHEFilter`, or `FFTFilter` fixes a rendered
  image (or a noisy/illumination-biased extraction) without touching the file on disk — filters
  live in `ImageSpec.filter_chain` and apply lazily, both at render time and at extraction time.

## 7. Xenium morphology image intensities

Xenium instruments capture a dedicated morphology image alongside the transcript data: a multichannel fluorescence image of the same section, used internally by the instrument's segmentation algorithm. Astrolabe exposes these per-cell as a separate `mdata['morphology']` modality, providing nuclear/cellular morphological stain intensities without any registration step (the image is already in the same coordinate frame as the transcript data).

`ab.datasets.xenium()` includes the morphology OME-TIFF (`morphology.ome.tif`, ~4.8 GB) as part of its download. Set `morphology_image=True` when loading, then call `io.extract_morphology_intensities` to populate `mdata['morphology'].X`. The result has the same `obs_names` as the Xenium modality and one column per morphology channel.

In [ ]:
import mudata as md

# Re-load the Xenium sample with morphology_image=True.
lib_id = list(mdata['xenium'].uns['spatial'])[0]
xenium_path = mdata['xenium'].uns['spatial'][lib_id]['source_path']

mdata_morph = ab.io.read_xenium(xenium_path, morphology_image=True)
ab.pipeline.filter_xenium(mdata_morph)

# Recreate the MuData after filtering so MuData's var_names tracking is reset
# before we add the new 'morphology' modality below.
mdata_morph = md.MuData(mdata_morph.mod)
mdata_morph.uns['default_modality'] = 'xenium'

print(mdata_morph['xenium'].uns['spatial'][lib_id]['images'].keys())

In [ ]:
ab.io.extract_morphology_intensities(mdata_morph) # TODO Make faster

print(mdata_morph['morphology'])
print()
print("morphology channels:", list(mdata_morph['morphology'].var_names))

The 11 channels are named `ch0`–`ch10` because the Xenium morphology OME-TIFF does not embed channel names in its metadata: the physical meaning of each channel (DAPI nuclear stain, antibody panel, etc.) depends on the specific Xenium panel and instrument configuration and must be looked up in the Xenium panel documentation or experiment manifest. Like the IF and H&E demos above, these intensities are real: the morphology image is already co-registered with the transcript data and at cellular resolution, so per-cell extractions are meaningful — and unlike either of those, no registration step was needed to get here, since this image comes from this exact Xenium run. A useful first downstream step is checking whether nuclear stain intensity (typically `ch0`) correlates with transcript count: a strong positive correlation would indicate that cell area or nuclear size is a confound in library-size normalization.

In [ ]:
# Intensity distributions across all morphology channels — useful for seeing which
# channels have strong nuclear vs. cytoplasmic signal.
ab.pl.intensity_histograms(mdata_morph['morphology']).show()

Distribution shape reveals each channel's biological content. A **bimodal distribution** (one peak near zero, background, and one positive peak) indicates genuine signal separating two cell populations. A **narrow unimodal distribution near zero** means the channel is below detection or absent from this panel configuration; such channels carry little discriminative information and can be dropped before downstream analysis. A **right-skewed, heavy-tailed distribution** is characteristic of nuclear stains: most cells have moderate intensity but compact, hyperchromatic cells (lymphocytes, mitotic tumor cells) push a long right tail. For the Xenium morphology channels, the channel-to-stain mapping (which `ch*` index is DAPI, which is a protein marker) must be looked up in the Xenium panel documentation: astrolabe uses generic `ch0`–`ch10` names because the OME-TIFF lacks embedded channel labels.

Like the IF panel above (but unlike the H&E scan, which needed a real affine from `he_imagealignment.csv`), the morphology image is already co-registered with the Xenium transcript data: no affine needed, so cell boundaries can be overlaid directly on the raw morphology image. The example below crops to a 500 µm × 500 µm window so rendering stays fast; the same call scales to the full tissue by removing the crop step.

In [ ]:
# Crop to a 500 µm × 500 µm sub-region for fast rendering.
crop_center = tuple(mdata_morph['xenium'].obsm['spatial'].mean(axis=0))
crop_rect = ab.pl.rectangle(crop_center, 500.0, 500.0)
morphology_crop = ab.pp.crop(mdata_morph, crop_rect, inplace=False)

# channel=0 is the first morphology channel (DAPI/nuclear stain in most Xenium panels).
(
    ab.pl.image(morphology_crop, image_key='morphology', channel=0)
    + ab.pl.boundaries(morphology_crop, alpha=0.6)
).show()